# 11 — Demo gợi ý Top-K từ một prefix

- Tải artifact đã lưu (vocabulary, metadata, cấu hình kNN tốt nhất trên validation, GRU4Rec đã train), fit trên **train** như biến thể `train_only` của notebook 10.
- Đầu vào: danh sách `(itemid, event, timestamp_ms)`. Các sự kiện cùng timestamp là một nhóm; thời điểm query là timestamp cuối.
- Đầu ra: bảng `rank, item_id, score, category_id, is_fallback`; `category_id` là danh mục gần nhất có `timestamp <= t` (thiếu thì `<NA>`).
- Ba ví dụ lấy từ test: kết quả phải **khớp** Top-K đã lưu từ lần chạy đánh giá (notebook 10, biến thể `train_only`) cho cùng query.

Chạy từ kernel mới: *Restart Kernel and Run All*, hoặc `jupyter nbconvert --to notebook --execute --inplace notebooks/11_demo.ipynb` sau khi đã chạy notebook 10.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display
from rr.io import fingerprint, read_json, require
from rr.demo import Recommender

final = read_json(ROOT / "reports/final/metrics.json")
require(final["split"] == "test" and not final["rehearsal"], "Cần lần chạy test thật của notebook 10.")
saved_info = final["saved_topk"]
require(fingerprint(ROOT / saved_info["path"])["sha256"] == saved_info["sha256"], "Top-K đã lưu khác report notebook 10.")
saved = pd.read_parquet(ROOT / saved_info["path"])
recommenders = {"knn": Recommender(ROOT, "knn"), "gru4rec": Recommender(ROOT, "gru4rec")}
print("kNN dùng trong demo:", recommenders["knn"].family)

## Một prefix tự nhập

In [ ]:
example = [(461686, "view", 1442004589439), (461686, "view", 1442004600000), (10572, "addtocart", 1442004600000)]
display(recommenders["knn"].recommend(example, k=10))

## Ba query test: so khớp với lần chạy đánh giá

In [ ]:
test_q = pd.read_parquet(ROOT / "data/processed/samples/test_queries.parquet")
events = pd.read_parquet(ROOT / "data/processed/events_sessionized.parquet",
                         columns=["session_id", "group_index", "itemid", "event", "timestamp", "event_id"])
rng = np.random.default_rng(2026)
picks = pd.concat([test_q[test_q["prefix_len_groups"] == 1].sample(1, random_state=1),
                   test_q[test_q["prefix_len_groups"].between(2, 4)].sample(1, random_state=2),
                   test_q[test_q["prefix_len_groups"] >= 5].sample(1, random_state=3)])
for _, q in picks.iterrows():
    ev = events[(events["session_id"] == q["session_id"]) & (events["group_index"] <= q["prefix_len_groups"])].sort_values(["timestamp", "event_id"])
    prefix = list(zip(ev["itemid"], ev["event"].astype(str), ev["timestamp"]))
    print(f"query {q['query_id']}: {q['prefix_len_groups']} nhóm, {len(prefix)} sự kiện")
    for kind, rec in recommenders.items():
        name = rec.family if kind == "knn" else "gru4rec"
        out = rec.recommend(prefix, k=20)
        ref = saved[(saved["model"] == name) & (saved["query_id"] == q["query_id"])].sort_values("rank")
        require(out["item_id"].tolist() == ref["itemid"].tolist() and out["is_fallback"].tolist() == ref["is_fallback"].tolist(),
                f"{name}: demo khác lần chạy đánh giá cho query {q['query_id']}.")
        print(f"  {name}: khớp lần chạy đánh giá ({len(out)} item)")
    display(out.head(10))